In [1]:
from IPython.display import display, HTML
display(HTML("""
<style>
div.container{width:85% !important;}
div.cell.code_cell.rendered{width:100%;}
div.input_prompt{padding:0px;}
div.CodeMirror {font-family:Consolas; font-size:20pt;}
div.output {font-size:12pt; font-weight:bold;}
div.input {font-family:Consolas; font-size:20pt;}
div.prompt {min-width:70px;}
div#toc-wrapper{padding-top:120px;}
div.text_cell_render ul li{font-size:12pt;padding:5px;}
table.dataframe{font-size:22px;}

</style>
"""))

## RAG절차

- 1.docx문서를 읽는다(python-docx) : \n은 제외
- 2.읽어온 문서를 쪼갠다.(tiktoken이용)
    * 모델의 context window와 비용확인
    * chunk(문서를 쪼갠 하나)가 길면 비용과 시간이 많이 소요
- 3. 쪼갠 문서를 임배딩 -> vector Store에 저장 => chroma(local vector Database / pinecorn(클라우드 vector Database)
- 4. 질문과 vector DB의 유사도 검색
- 5. 유사도 검색으로 가져온 문서

# 1. 문서를 읽는다.(python-docx)
- pip install -U -q langchain-community python-docs docx2txt 

In [2]:
%pip install -U -q langchain-community docx2txt python-docx

Note: you may need to restart the kernel to use updated packages.


In [4]:
# 이렇게 docx파일을 읽어오면 한꺼번에 가져옴(\n포함)
from langchain_community.document_loaders import Docx2txtLoader
loader = Docx2txtLoader('./tex_docx/소득세법(법률)(제21221호)(20260701).docx')
document = loader.load()
len(document)

1

In [10]:
# 한 문단씩 가져오기
from docx import Document
documents = Document('./tex_docx/소득세법(법률)(제21221호)(20260701)_.docx')
print(document)
print(dir(document))

['_Document__body', '__class__', '__delattr__', '__dict__', '__dir__', '__doc__', '__eq__', '__format__', '__ge__', '__getattribute__', '__gt__', '__hash__', '__init__', '__init_subclass__', '__le__', '__lt__', '__module__', '__ne__', '__new__', '__reduce__', '__reduce_ex__', '__repr__', '__setattr__', '__sizeof__', '__str__', '__subclasshook__', '__weakref__', '_block_width', '_body', '_element', '_parent', '_part', 'add_comment', 'add_heading', 'add_page_break', 'add_paragraph', 'add_picture', 'add_section', 'add_table', 'comments', 'core_properties', 'element', 'inline_shapes', 'iter_inner_content', 'paragraphs', 'part', 'save', 'sections', 'settings', 'styles', 'tables']


In [13]:
len(documents.paragraphs)
for paragrphs in documents.paragraphs[:10]:
    print('-------------------')
    print(paragrphs.text)

-------------------

-------------------
소득세법
-------------------
[시행 2026. 7. 1.] [법률 제21221호, 2025. 12. 23., 일부개정]
-------------------
재정경제부(재산세제과(양도소득세)) 044-215-4312
-------------------
재정경제부(소득세제과(근로소득)) 044-215-4216
-------------------
재정경제부(금융세제과(이자소득, 배당소득)) 044-215-4233
-------------------
재정경제부(소득세제과(사업소득, 기타소득)) 044-215-4217
-------------------
재정경제부(국제조세제도과(비거주자)) 044-215-4651
-------------------

-------------------
제1장 총칙 <개정 2009. 12. 31.>


In [16]:
# \n이 제거된 문서 전체 내용을 full_text
full_text =""
for paragraph in documents.paragraphs:
    full_text += paragraph.text +" "
full_text[:100]

' 소득세법 [시행 2026. 7. 1.] [법률 제21221호, 2025. 12. 23., 일부개정] 재정경제부(재산세제과(양도소득세)) 044-215-4312 재정경제부(소득세제'

# 2. 문서를 쪼갠다.
- pip install tiktoken : openAI의 공식 토크나이저 라이브러리
    - full_text -> 토큰단위로 쪼개서 숫자
    - 숫자-> 문자
    - 1500(?) 토큰씩 문서를 쪼개기

In [19]:
import tiktoken
encoder = tiktoken.encoding_for_model('gpt-4o-mini')
decoded = encoder.encode(full_text)

In [21]:
print(f"글자수 : {len(full_text)}")
print(f"토큰수 : {len(encoding)}")
print('문자들:',decoded[:10])
print('숫자들:',encoding[:10])

글자수 : 248301
토큰수 : 162419
문자들: [22183, 64328, 11734, 22070, 723, 5637, 15719, 220, 1323, 21]
숫자들: [22183, 64328, 11734, 22070, 723, 5637, 15719, 220, 1323, 21]


In [23]:
print(encoder.encode('Hello'),encoder.encode('안녕'),encoder.encode('친구'))

[13225] [14307, 98931] [61407, 10997]


In [24]:
print(list(range(0,162419,1500)))

[0, 1500, 3000, 4500, 6000, 7500, 9000, 10500, 12000, 13500, 15000, 16500, 18000, 19500, 21000, 22500, 24000, 25500, 27000, 28500, 30000, 31500, 33000, 34500, 36000, 37500, 39000, 40500, 42000, 43500, 45000, 46500, 48000, 49500, 51000, 52500, 54000, 55500, 57000, 58500, 60000, 61500, 63000, 64500, 66000, 67500, 69000, 70500, 72000, 73500, 75000, 76500, 78000, 79500, 81000, 82500, 84000, 85500, 87000, 88500, 90000, 91500, 93000, 94500, 96000, 97500, 99000, 100500, 102000, 103500, 105000, 106500, 108000, 109500, 111000, 112500, 114000, 115500, 117000, 118500, 120000, 121500, 123000, 124500, 126000, 127500, 129000, 130500, 132000, 133500, 135000, 136500, 138000, 139500, 141000, 142500, 144000, 145500, 147000, 148500, 150000, 151500, 153000, 154500, 156000, 157500, 159000, 160500, 162000]


In [38]:
def split_text(full_text=full_text,chunk_size=1500):
    'full_text를 chunk_size 토큰만큼 쪼개서 chunk_list로 반환'
    encoder = tiktoken.encoding_for_model('gpt-4o-mini')# nano 모델은 안됨
    total_encoding = encoder.encode(full_text) # full_text를 토큰 단위로 숫자 list
    total_token_count = len(total_encoding) # full_text의 토큰수 
    chunk_list = []
    for i in range(0,total_token_count,chunk_size):
        chunk_encoding = total_encoding[i:i+chunk_size]
        chunk_encoded = encoder.decode(chunk_encoding)
        #print(chunk_encoding,'=>',chunk_encoded)
        chunk_list.append(chunk_encoded)
    return chunk_list

In [39]:
example_text = "소득세법 법률 일부 개정함 이자소득 배당소득"
split_text(full_text = example_text, chunk_size=10)

['소득세법 법률 일부 개정함', ' 이자소득 배당소득']

In [40]:
chunk_list = split_text(full_text,1500)
len(chunk_list)

109

# 3. 쪼갠 문서를 임배딩 -> vector Store(인메모리)에 저장
- pip install chromadb

In [43]:
%pip install -q chromadb --no-warn-script-location

Note: you may need to restart the kernel to use updated packages.


In [58]:
import chromadb
chroma_client = chromadb.Client()

In [63]:
# collection은 RDB의 태이블 개념
collection_name = 'tax_collection'

In [64]:
from dotenv import load_dotenv
from chromadb.utils.embedding_functions import OpenAIEmbeddingFunction
load_dotenv()
import os
openai_key = os.getenv('OPENAI_API_KEY')
openai_embedding = OpenAIEmbeddingFunction(
    api_key = openai_key,
    model_name = 'text-embedding-3-large'
)

In [65]:
# collection에 입력할떄 사용될 id들(문자)
ids=[str(id) for id in range(len(chunk_list))]
print(ids)

['0', '1', '2', '3', '4', '5', '6', '7', '8', '9', '10', '11', '12', '13', '14', '15', '16', '17', '18', '19', '20', '21', '22', '23', '24', '25', '26', '27', '28', '29', '30', '31', '32', '33', '34', '35', '36', '37', '38', '39', '40', '41', '42', '43', '44', '45', '46', '47', '48', '49', '50', '51', '52', '53', '54', '55', '56', '57', '58', '59', '60', '61', '62', '63', '64', '65', '66', '67', '68', '69', '70', '71', '72', '73', '74', '75', '76', '77', '78', '79', '80', '81', '82', '83', '84', '85', '86', '87', '88', '89', '90', '91', '92', '93', '94', '95', '96', '97', '98', '99', '100', '101', '102', '103', '104', '105', '106', '107', '108']


In [66]:
# chromDB(인메모리 vector Store==vector Datebase)에 collection 생성
tax_collection = chroma_client.get_or_create_collection(
                    name=collection_name,
                    embedding_function=openai_embedding
)

In [67]:
tax_collection.add(documents=chunk_list,ids=ids)

In [69]:
#tax_collection
results = tax_collection.get(include=['embeddings','documents','metadatas'])
print('전체 chunk수 : ', len(results['ids']))
print('ID목록(처음3개까지만):', results['ids'][:5])
print('ID목록(처음3개까지만):', results['embeddings'][:5])
print('ID목록(처음3개까지만):', results['documents'][0][:20])
print('ID목록(처음3개까지만):', results['metadatas'][:5])

전체 chunk수 :  109
ID목록(처음3개까지만): ['0', '1', '2', '3', '4']
ID목록(처음3개까지만): [[ 0.0244751  -0.02638245  0.00180626 ... -0.00321579 -0.01422882
  -0.019104  ]
 [ 0.03497314 -0.01469421 -0.00390434 ... -0.00169182 -0.02319336
  -0.01204681]
 [ 0.05560303  0.00403214 -0.00764084 ... -0.0011549  -0.0045166
  -0.0154953 ]
 [ 0.03567505 -0.03616333 -0.00316048 ... -0.00646973 -0.02113342
  -0.01062012]
 [ 0.0125885  -0.026474   -0.00330925 ... -0.00468445 -0.01100922
  -0.01321411]]
ID목록(처음3개까지만):  소득세법 [시행 2026. 7. 1
ID목록(처음3개까지만): [None, None, None, None, None]


# 4.질문과 vectorDB의 유사도 검색

In [70]:
query = "연봉 5천만원인 직장인의ㅏ 소득세는 얼마인가요?"
retrieved_doc = tax_collection.query(query_texts=query,
                                    n_results=3 # 유사도가 높은 상위 3개 추출(기본값은 10개)
                                    )

In [75]:
len(retrieved_doc['documents'][0])


3

# 5. 유사도 검색으로 나온 문서를 LLM에 질문과 같이 전달

In [76]:
from openai import OpenAI
load_dotenv()
client = OpenAI()
response = client.chat.completions.create(
    model= 'gpt-4o-mini',
    messages=[
        {'role':'system',
         'content':f'''당신은 한국 소득세 전문가입니다.아래의 내용을 참고해서 사용자 질문에 답변해주세요.
         {retrieved_doc['documents'][0]}'''            
        },
        {'role':'user','content':query}
    ]
)

In [77]:
print(response.choices[0].message.content)

연봉 5천만원인 직장인의 소득세를 계산하기 위해서는 아래와 같은 단계로 진행합니다.

1. **과세표준 계산**: 한국에서는 기본적으로 연봉에서 각종 공제를 차감한 후 과세표준을 계산합니다. 일반적으로 직장인은 기본공제, 인적공제, 그리고 근로소득 공제를 고려합니다.

2. **근로소득 세액 공제**: 근로소득자에게는 근로소득 세액 공제도 있습니다. 이는 과세된 소득에 따라 다르게 적용됩니다.

3. **세액 계산**: 과세표준에 대한 기본 세율에 따라 세액을 계산합니다. 한국의 소득세는 누진세율 구조이므로 과세표준이 올라갈수록 세율이 높아집니다.

예를 들어, 2023년도 기준으로 연봉 5천만원에 대한 소득세를 대략적으로 계산해보면 다음과 같습니다:

1. **총 합계 소득**: 5,000만원
2. **근로소득 공제**: 일반적으로 약간의 비율로 계산됩니다. 대략적으로 2,300만원 정도 공제
3. **과세표준**: 5,000만원 - 2,300만원 = 2,700만원
4. **소득세율 계산**: 대략적으로 누진세율에 의해 계산
   - 1,200만원까지는 6%
   - 1,200만원~4,600만원 구간의 차액에는 15%가 적용됨.

이러한 방법으로 대략적인 세액을 계산하면 약 200만원 정도의 소득세가 나올 수 있습니다. 

정확한 계산은 개인의 상황에 따라 다를 수 있으니, 세무사나 관련 전문가와 상담하는 것도 좋은 방법입니다.
